In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))

import math
import numpy as np
import torch
import torchvision

from sklearn.decomposition import PCA

from datasets.mlretset import MLRetSet
from metrics import psnr, sa

In [ ]:
dataset_path = "./datasets/MLRetSet/"

In [ ]:
test_dataset = MLRetSet(dataset_path, split="test", transform=torchvision.transforms.CenterCrop(96))
src_channels = 369

psnr_metric = psnr.PeakSignalToNoiseRatio()
sa_metric = sa.SpectralAngle()

In [ ]:
for cratio in [4, 8, 16, 32, 64, 128, 256, 512]:
    cr_list = []
    psnr_list = []
    sa_list = []

    latent_channels = int(math.ceil(src_channels/cratio))
    pca = PCA(n_components=latent_channels)

    for data_org in test_dataset:
        data_in = data_org
        data_in = data_in.flatten(1, 2)
        data_in = data_in.T
        data_in = data_in.numpy()

        data_lat = pca.fit_transform(data_in)

        data_out = pca.inverse_transform(data_lat)
        data_out = data_out.T
        data_out = np.resize(data_out, (369, 96, 96))

        data_rec = torch.from_numpy(data_out)

        size_raw = data_in.nbytes

        size_lat = data_lat.nbytes

        org_size = 202 * 128 * 128
        lat_size = latent_channels * 128 * 128 + src_channels * latent_channels + src_channels # pca coefficients + pca basis + mean vector
        cr = org_size / lat_size 

        psnr = psnr_metric(data_org, data_rec)

        sa = sa_metric(data_org, data_rec)

        cr_list.append(cr)
        psnr_list.append(psnr)
        sa_list.append(sa)

    cr_avg = np.mean(cr_list)
    psnr_avg = np.mean(psnr_list)
    sa_avg = np.nanmean(sa_list)

    print(latent_channels)
    print(f"cr_avg:\t\t{cr_avg}")
    print(f"psnr_avg:\t{psnr_avg} dB")
    print(f"sa_avg: \t{sa_avg} °")
    print("===")